# **At first, you should have ChEMBL36 database at your local hard drive.**

In [1]:
import collections
import pandas as pd
import matplotlib.pyplot as plt
import psycopg

/home/linu/anaconda3/envs/postgresql/lib/python3.12/site-packages/apache_age_python-0.0.7-py3.12.egg/age/age.py:29: SyntaxWarning: invalid escape sequence '\s'
/home/linu/anaconda3/envs/postgresql/lib/python3.12/site-packages/apache_age_python-0.0.7-py3.12.egg/age/age.py:29: SyntaxWarning: invalid escape sequence '\s'


In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit import DataStructs
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import MACCSkeys
import pandas as pd
import numpy as np
import sys

In [3]:
# Connection parameters - update user, password, host, port as needed
conn_params = {
    "dbname": "chembl_36",
    "user": "postgres",
    "password": "123456",
    "host": "localhost",
    "port": 5432
}

In [4]:
%%time
# Connect to the PostgreSQL database
with psycopg.connect(**conn_params) as conn:
    with conn.cursor() as cur:
        # Query to join or fetch columns separately, here assuming simple cross join example
        query = """
        SELECT cs.canonical_smiles, md.chembl_id
        FROM compound_structures cs
        JOIN molecule_dictionary md ON cs.molregno = md.molregno
        """
        cur.execute(query)
        # Fetch all records
        records = cur.fetchall()
        
        # Convert to pandas DataFrame
        df = pd.DataFrame(records, columns=["canonical_smiles", "chembl_id"])

conn.close()

CPU times: user 8.86 s, sys: 357 ms, total: 9.22 s
Wall time: 10.7 s


In [5]:
df

,canonical_smiles,chembl_id
0,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl,CHEMBL6329
1,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1,CHEMBL6328
2,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1,CHEMBL265667
3,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1,CHEMBL6362
4,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1,CHEMBL267864
...,...,...
2854810,N[C@@H]1[C@H](Nc2ccc3ccc(C(=O)Nc4ccccc4Cl)n3n2...,CHEMBL6071105
2854811,Cc1ccccc1-c1c[n+](C)c(-c2ccc(C(C)(C)C)cc2)cc1/...,CHEMBL6071106
2854812,CCN(CC)c1ccc2c(c1)OC1=C(/C=C/C3=[N+](C)c4ccccc...,CHEMBL6071107
2854813,C[C@@H]1[C@H](O)[C@@H](C)/C=C/C=C/C=C/C=C/C=C/...,CHEMBL6071108


In [6]:
len(df)

2854815

In [5]:
from rdkit.Chem import SaltRemover as sr
remover = sr.SaltRemover()

In [9]:
df1 = df.isna()['canonical_smiles']
df1

0          False
1          False
2          False
3          False
4          False
           ...  
2409265    False
2409266    False
2409267    False
2409268    False
2409269    False
Name: canonical_smiles, Length: 2409270, dtype: bool

In [10]:
from tqdm import tqdm

In [11]:
k = 0
for i in tqdm(range(len(df1))):
    if df1[i] == True:
        k += 1
        #print(i)
print('number of null in smiles:', k)

100%|█████████████████████████████| 2409270/2409270 [00:07<00:00, 316856.22it/s]

number of null in smiles: 0


In [15]:
chemblid_list = [df['chembl_id'][i] for i in tqdm(range(len(df['chembl_id'])))]

100%|█| 2409270/2409270 [00:09<00:00, 2


In [13]:
#Generating MORGAN
import sys
import tqdm
import os
import contextlib
#To suppress the output written messages. 
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

indices_to_remove = []
FPs = []
fpg = AllChem.GetMorganGenerator(radius=3, fpSize=1024) 
uncharger = rdMolStandardize.Uncharger()

for i in tqdm.tqdm(range(len(df['canonical_smiles']))):
    try:
        with suppress_stdout_stderr():
            mol_i = Chem.MolFromSmiles(df['canonical_smiles'][i])
            neutral_mol = uncharger.uncharge(mol_i)
            Chem.SanitizeMol(neutral_mol)
            fp = fpg.GetFingerprint(neutral_mol)
        FPs.append(fp)
    except Exception as e:
        tqdm.tqdm.write(f"An error occurred for molecule {i} as: {e}")
        indices_to_remove.append(i)


  7%|▎    | 198277/2854815 [01:01<13:41, 3233.41it/s]

An error occurred for molecule 197770 as: Python argument types in
    Uncharger.uncharge(Uncharger, NoneType)
did not match C++ signature:
    uncharge(RDKit::MolStandardize::Uncharger {lvalue} self, RDKit::ROMol mol)


 21%|█    | 590008/2854815 [03:07<14:47, 2552.43it/s]

An error occurred for molecule 589660 as: Python argument types in
    Uncharger.uncharge(Uncharger, NoneType)
did not match C++ signature:
    uncharge(RDKit::MolStandardize::Uncharger {lvalue} self, RDKit::ROMol mol)


 64%|██▌ | 1834024/2854815 [09:26<06:07, 2775.64it/s]

...

 81%|███▎| 2324986/2854815 [12:08<03:25, 2577.43it/s]

An error occurred for molecule 2324701 as: Python argument types in
    Uncharger.uncharge(Uncharger, NoneType)
did not match C++ signature:
    uncharge(RDKit::MolStandardize::Uncharger {lvalue} self, RDKit::ROMol mol)


 92%|███▋| 2615291/2854815 [13:54<01:29, 2668.38it/s]

An error occurred for molecule 2614840 as: Python argument types in
    Uncharger.uncharge(Uncharger, NoneType)
did not match C++ signature:
    uncharge(RDKit::MolStandardize::Uncharger {lvalue} self, RDKit::ROMol mol)


100%|████| 2854815/2854815 [15:23<00:00, 3091.46it/s]


In [14]:
len(indices_to_remove)

17

In [26]:
len(FPs), len(df)

(2854798, 2854815)

In [28]:
indices_to_remove = [197828, 589796, 1834036, 1938062, 1938066, 1938069, 1938079, 1938099, 1938101, 2312135, 2312427, 2313367, 2313963, 2318519, 2321673, 2325346]

In [29]:
print(indices_to_remove)

[197828, 589796, 1834036, 1938062, 1938066, 1938069, 1938079, 1938099, 1938101, 2312135, 2312427, 2313367, 2313963, 2318519, 2321673, 2325346]


In [ ]:
# By running this cell, molecules with error are removed from the list of chemblid_list.
# So, lenght of the FPs and chemblid_list will be equal.
for index in sorted(indices_to_remove, reverse=True):
    del chemblid_list[index]
len(chemblid_list)

In [34]:
# To save the FPs as a numpy fromat
from rdkit.DataStructs import ConvertToNumpyArray

# Create empty NumPy array shape (num_fps, fp_size)
num_fps = len(FPs)
fp_size = 1024  # your fingerprint size
arr = np.zeros((num_fps, fp_size), dtype=np.uint8)

for i, fp in enumerate(FPs):
    ConvertToNumpyArray(fp, arr[i])

# Save compressed .npz file
np.savez_compressed('chembl36_FPs.npz', fingerprints=arr)
